# RAG Avancé – RGPD
## Architecture : Multi-Query → Parent Document Retriever → Re-ranker → LLM

```
Question utilisateur
        │
Multi-Query Retriever  (reformule en N variantes)
        │
Vector Store (chunks)  +  métadonnées (chapitre, titre, article)
        │
Parent Document Retriever  (renvoie l'article complet au LLM)
        │
Re-ranker  (cross-encoder pour trier les articles)
        │
LLM HuggingFace (Qwen2.5-0.5B-Instruct)
        │
Réponse structurée + articles cités
```


## 1. Imports & dépendances

```bash
pip install langchain langchain-community langchain-chroma chromadb sentence-transformers transformers accelerate beautifulsoup4 requests
```

In [1]:
import requests
import json
import re
import uuid

from bs4 import BeautifulSoup
from dotenv import load_dotenv

load_dotenv()


True

## 2. Chargement et parsing du RGPD

On extrait chaque article avec ses métadonnées : numéro de chapitre, titre du chapitre, numéro d'article, titre d'article.

In [2]:
URL = "https://eur-lex.europa.eu/legal-content/FR/TXT/HTML/?uri=CELEX:32016R0679"

response = requests.get(URL, headers={"Accept-Language": "fr"})
response.encoding = "utf-8"

soup = BeautifulSoup(response.text, "html.parser")


def clean_text(element):
    if element is None:
        return ""
    return element.get_text(" ", strip=True)


resultat = []

# Vrais chapitres : cpt_I, cpt_II, cpt_III…
chapitres = soup.find_all("div", id=re.compile(r"^cpt_[IVXLCDM]+$"))

for chapitre in chapitres:
    chapitre_nom   = clean_text(chapitre.find("p", class_="oj-ti-section-1"))
    titre_chapitre = clean_text(chapitre.find("p", class_="oj-ti-section-2"))

    if not chapitre_nom.startswith("CHAPITRE"):
        continue

    chapitre_data = {
        "chapitre":       chapitre_nom,
        "titre_chapitre": titre_chapitre,
        "contenu":        []
    }

    articles         = chapitre.find_all("div", id=re.compile(r"^art_\d+$"))
    articles_deja_vus = set()

    for article in articles:
        article_id = article.get("id")
        if article_id in articles_deja_vus:
            continue
        articles_deja_vus.add(article_id)

        numero_article = clean_text(article.find("p", class_="oj-ti-art"))
        titre_article  = clean_text(article.find("p", class_="oj-sti-art"))

        if not numero_article.startswith("Article"):
            continue

        paragraphes     = article.find_all("p", class_="oj-normal")
        contenu_article = "\n".join(
            clean_text(p) for p in paragraphes if clean_text(p)
        )

        chapitre_data["contenu"].append({
            "article":         numero_article,
            "titre_article":   titre_article,
            "contenu_article": contenu_article
        })

    resultat.append(chapitre_data)

print(f"✅  {len(resultat)} chapitres extraits")
print(f"✅  {sum(len(c['contenu']) for c in resultat)} articles extraits")

with open("rgpd_structure.json", "w", encoding="utf-8") as f:
    json.dump(resultat, f, ensure_ascii=False, indent=4)


✅  11 chapitres extraits
✅  99 articles extraits


## 3. Construction des Documents LangChain avec métadonnées

### Stratégie Parent / Child
- **Child chunks** (small) : utilisés pour la recherche vectorielle (chunk_size=300, overlap=50)
- **Parent documents** : l'article complet, renvoyé au LLM pour la génération

Chaque chunk enfant porte les métadonnées :
```python
{
  "chapitre":         "CHAPITRE II",
  "titre_chapitre":   "Principes",
  "article":          "Article 5",
  "titre_article":    "Principes relatifs au traitement des données",
  "type":             "child_chunk",
  "parent_id":        "<uuid>",   # lien vers l'article complet
  "source":           "RGPD"
}
```


In [6]:
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter

with open("rgpd_structure.json", "r", encoding="utf-8") as f:
    data = json.load(f)

# ── Splitter pour les child chunks ──────────────────────────────────
# chunk_size=300 : assez grand pour capturer le sens d'un paragraphe RGPD
# chunk_overlap=50 : évite de couper une phrase entre deux chunks
child_splitter = RecursiveCharacterTextSplitter(
    chunk_size=300,
    chunk_overlap=50,
    separators=["\n\n", "\n", ". ", " ", ""]
)

parent_documents = []   # articles complets → envoyés au LLM
child_documents  = []   # petits chunks     → indexés dans le vector store

for chapitre in data:
    chapitre_nom   = chapitre.get("chapitre", "")
    titre_chapitre = chapitre.get("titre_chapitre", "")

    for article in chapitre.get("contenu", []):
        numero_article  = article.get("article", "")
        titre_article   = article.get("titre_article", "")
        contenu_article = article.get("contenu_article", "")

        if not contenu_article.strip():
            continue

        # Texte complet de l'article (parent)
        parent_text = (
            f"{chapitre_nom} – {titre_chapitre}\n"
            f"{numero_article} – {titre_article}\n\n"
            f"{contenu_article}"
        )
        parent_id = str(uuid.uuid4())

        # Métadonnées partagées
        metadata_base = {
            "chapitre":         chapitre_nom,
            "titre_chapitre":   titre_chapitre,
            "article":          numero_article,
            "titre_article":    titre_article,
            "source":           "RGPD",
            "parent_id":        parent_id,
        }

        # Document parent (article complet)
        parent_doc = Document(
            page_content=parent_text,
            metadata={**metadata_base, "type": "parent"}
        )
        parent_documents.append(parent_doc)

        # Child chunks (pour la recherche)
        chunks = child_splitter.split_text(contenu_article)
        for i, chunk in enumerate(chunks):
            child_doc = Document(
                page_content=chunk,
                metadata={**metadata_base, "type": "child_chunk", "chunk_index": i}
            )
            child_documents.append(child_doc)

print(f"✅  {len(parent_documents)} articles parents")
print(f"✅  {len(child_documents)} child chunks créés")
print()
print("Exemple de métadonnées d'un child chunk :")
print(json.dumps(child_documents[0].metadata, ensure_ascii=False, indent=2))


c:\Users\berul\OneDrive\Bureau\AI31 Projet\AI31_Projet_RAG\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✅  99 articles parents
✅  1067 child chunks créés

Exemple de métadonnées d'un child chunk :
{
  "chapitre": "CHAPITRE I",
  "titre_chapitre": "Dispositions générales",
  "article": "Article premier",
  "titre_article": "Objet et objectifs",
  "source": "RGPD",
  "parent_id": "693b872e-0d5e-4864-88ff-e172e892e051",
  "type": "child_chunk",
  "chunk_index": 0
}


## 4. Embeddings & Vector Store

On indexe uniquement les **child chunks** dans ChromaDB.
Le modèle `paraphrase-multilingual-MiniLM-L12-v2` est choisi car il supporte nativement le français (bien meilleur que `all-MiniLM-L6-v2` pour du RGPD).

In [7]:
from langchain_community.vectorstores import Chroma
from langchain_community.embeddings import HuggingFaceEmbeddings

# Modèle multilingue : bien adapté au français juridique
embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2",
    model_kwargs={"device": "cpu"}
)

# On stocke les child chunks dans Chroma
vector_store = Chroma.from_documents(
    documents=child_documents,
    embedding=embedding_model,
    collection_name="rgpd_chunks"
)

# Index parent_id → parent_document pour récupération rapide
parent_index = {doc.metadata["parent_id"]: doc for doc in parent_documents}

print(f"✅  Vector store créé avec {vector_store._collection.count()} chunks")


C:\Users\berul\AppData\Local\Temp\ipykernel_102492\2607370998.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import Chroma
C:\Users\berul\AppData\Local\Temp\ipykernel_102492\2607370998.py:5: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embedding_model = HuggingFaceEmbeddings(
c:\Users\berul\OneDrive\Bureau\AI31 Projet\AI31_Projet_RAG\venv\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses s

✅  Vector store créé avec 1067 chunks


## 5. Multi-Query Retriever

Le LLM reformule la question en **4 variantes** pour couvrir plus d'angles sémantiques.
Exemple :
```
Question : "quelles obligations pour traiter des emails ?"
→ "base légale du traitement de données personnelles"
→ "email données personnelles RGPD obligations"
→ "information des personnes concernées traitement"
→ "minimisation des données collecte email"
```


In [8]:
from langchain_community.llms import HuggingFacePipeline
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline

# ── LLM pour la reformulation Multi-Query ───────────────────────────
# Qwen2.5-0.5B-Instruct : léger, gratuit, instruct-tuned
model_name = "Qwen/Qwen2.5-0.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model_hf  = AutoModelForCausalLM.from_pretrained(model_name)

pipe = pipeline(
    "text-generation",
    model=model_hf,
    tokenizer=tokenizer,
    max_new_tokens=256,
    temperature=0.3,
    do_sample=True,
)

llm = HuggingFacePipeline(pipeline=pipe)


Loading weights: 100%|██████████| 290/290 [00:00<00:00, 9879.45it/s]
[transformers] Passing `generation_config` together with generation-related arguments=({'do_sample', 'temperature', 'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
C:\Users\berul\AppData\Local\Temp\ipykernel_102492\1362788119.py:20: LangChainDeprecationWarning: The class `HuggingFacePipeline` was deprecated in LangChain 0.0.37 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFacePipeline``.
  llm = HuggingFacePipeline(pipeline=pipe)


In [13]:
from langchain_experimental.retrievers import MultiQueryRetriever
from langchain_core.prompts import PromptTemplate

# Prompt de reformulation en français
multi_query_prompt = PromptTemplate(
    input_variables=["question"],
    template="""Tu es un assistant spécialisé en droit RGPD.
Ta tâche est de reformuler la question suivante en 4 variantes différentes,
pour améliorer la recherche dans une base documentaire juridique.
Génère uniquement les 4 questions, une par ligne, sans numérotation.

Question originale : {question}
Variantes :"""
)

# Retriever de base (sur les child chunks)
base_retriever = vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 6}
)

# --- Adaptation LLM local → ChatModel ---
from langchain_core.language_models import BaseLanguageModel

class SimpleLLM(BaseLanguageModel):
    def __init__(self, pipeline):
        super().__init__()
        self.pipeline = pipeline

    def invoke(self, prompt, **kwargs):
        return self.pipeline(prompt)[0]["generated_text"]

llm_chat = SimpleLLM(llm.pipeline)

# Multi-Query Retriever
multi_query_retriever = MultiQueryRetriever.from_llm(
    retriever=base_retriever,
    llm=llm_chat,
    prompt=multi_query_prompt
)

print("✅ Multi-Query Retriever prêt")


ImportError: cannot import name 'MultiQueryRetriever' from 'langchain_experimental.retrievers' (c:\Users\berul\OneDrive\Bureau\AI31 Projet\AI31_Projet_RAG\venv\Lib\site-packages\langchain_experimental\retrievers\__init__.py)

In [14]:
from langchain_core.prompts import PromptTemplate

# Prompt de reformulation en français
multi_query_prompt = PromptTemplate(
    input_variables=["question"],
    template="""Tu es un assistant spécialisé en droit RGPD.
Ta tâche est de reformuler la question suivante en 4 variantes différentes,
pour améliorer la recherche dans une base documentaire juridique.
Génère uniquement les 4 questions, une par ligne, sans numérotation.

Question originale : {question}
Variantes :"""
)

# Retriever de base (sur les child chunks)
base_retriever = vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 6}
)

# --- Fonction pour générer les reformulations ---
def generate_queries(question: str, llm, prompt):
    reformulations = llm.invoke(
        prompt.format(question=question)
    )
    # Nettoyage : une ligne = une requête
    return [q.strip() for q in reformulations.split("\n") if q.strip()]

# --- Fonction Multi‑Query moderne ---
def multi_query_search(question: str, retriever, llm, prompt):
    # 1. Générer 4 variantes
    queries = generate_queries(question, llm, prompt)

    all_docs = []
    for q in queries:
        docs = retriever.invoke(q)
        all_docs.extend(docs)

    # 2. Déduplication par contenu
    unique = {doc.page_content: doc for doc in all_docs}
    return list(unique.values())

print("✅ Multi‑Query moderne prêt")


✅ Multi‑Query moderne prêt


## 6. Parent Document Retriever

Après la recherche sur les child chunks, on remonte les **articles complets** (parents).
- Les child chunks donnent les meilleurs matches sémantiques
- Mais on envoie au LLM l'article entier → pas de perte de contexte juridique


In [18]:
def retrieve_parent_documents(question: str, retriever, parent_idx: dict, llm=None, prompt=None, top_k: int = 5):
    """
    1. Cherche les child chunks pertinents via multi_query_search()
    2. Remonte les articles parents correspondants (dédupliqués)
    3. Retourne au max top_k articles parents uniques
    """

    # --- 1. Multi‑query search sur les child chunks ---
    child_hits = multi_query_search(
        question=question,
        retriever=retriever,
        llm=llm,
        prompt=prompt
    )

    # --- 2. Remonter les documents parents ---
    seen_parent_ids = set()
    parent_docs = []

    for child in child_hits:
        pid = child.metadata.get("parent_id")
        if pid and pid not in seen_parent_ids:
            seen_parent_ids.add(pid)
            parent_doc = parent_idx.get(pid)
            if parent_doc:
                parent_docs.append(parent_doc)

        if len(parent_docs) >= top_k:
            break

    return parent_docs

test_docs = retrieve_parent_documents(
    question="Quelles sont les bases légales du traitement ?",
    retriever=base_retriever,
    parent_idx=parent_index,
    llm=llm,
    prompt=multi_query_prompt
)

print(f"✅  {len(test_docs)} articles parents récupérés")
for d in test_docs:
    meta = d.metadata
    print(f"   • {meta['article']} | {meta['chapitre']} – {meta['titre_chapitre']}")


[transformers] Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer Qwen2Tokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


✅  5 articles parents récupérés
   • Article 30 | CHAPITRE IV – Responsable du traitement et sous-traitant
   • Article 38 | CHAPITRE IV – Responsable du traitement et sous-traitant
   • Article 28 | CHAPITRE IV – Responsable du traitement et sous-traitant
   • Article 56 | CHAPITRE VI – Autorités de contrôle indépendantes
   • Article 60 | CHAPITRE VII – Coopération et cohérence


## 7. Re-ranker (Cross-Encoder)

Le re-ranker utilise un **cross-encoder** : il évalue la pertinence de chaque article
par rapport à la question (bien plus précis qu'un cosine similarity).

Modèle : `cross-encoder/ms-marco-MiniLM-L-6-v2` — léger et efficace.


In [19]:
from sentence_transformers import CrossEncoder

# Cross-encoder pour le re-ranking
reranker = CrossEncoder(
    "cross-encoder/ms-marco-MiniLM-L-6-v2",
    max_length=512
)


def rerank_documents(question: str, docs: list, top_n: int = 3):
    """
    Score chaque article avec le cross-encoder,
    retourne les top_n articles les plus pertinents.
    """
    if not docs:
        return []

    # Paires (question, contenu de l'article)
    pairs = [(question, doc.page_content[:512]) for doc in docs]

    scores = reranker.predict(pairs)

    # Tri décroissant par score
    ranked = sorted(zip(scores, docs), key=lambda x: x[0], reverse=True)

    return [doc for _, doc in ranked[:top_n]]


# Test
reranked = rerank_documents(
    "Quelles sont les bases légales du traitement ?",
    test_docs
)

print(f"✅  {len(reranked)} articles après re-ranking")
for d in reranked:
    meta = d.metadata
    print(f"   • {meta['article']} – {meta['titre_article']}")


Loading weights: 100%|██████████| 105/105 [00:00<00:00, 14598.31it/s]


✅  3 articles après re-ranking
   • Article 38 – Fonction du délégué à la protection des données
   • Article 28 – Sous-traitant
   • Article 30 – Registre des activités de traitement


## 8. Prompt RGPD & RAG Chain finale

### Pipeline complet
```
question
  → Multi-Query Retriever (child chunks)
  → Parent Document Retriever (articles complets)
  → Re-ranker (top 3 articles)
  → LLM + prompt structuré
  → Réponse avec articles cités
```


In [23]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableLambda

# ── Prompt structuré RGPD ────────────────────────────────────────────
RGPD_PROMPT = """Tu es un assistant expert en conformité RGPD.
Tu réponds UNIQUEMENT à partir des articles RGPD fournis dans le contexte.
Si l'information n'est pas présente, dis : "Je ne peux pas répondre avec certitude à partir des extraits RGPD fournis."

Contexte RGPD (articles complets) :
{context}

Question : {question}

Réponds en suivant EXACTEMENT cette structure :

## 1. Réponse directe
[Réponse claire et concise à la question]

## 2. Articles RGPD concernés
[Liste des articles et chapitres présents dans le contexte]

## 3. Explication détaillée
[Explique ce que disent les articles pertinents]

## 4. Points de vigilance
[Base légale, consentement, minimisation, durée de conservation, sécurité, données sensibles]

## 5. Recommandation pratique
[Conseil concret et actionnable]

⚠️ Avertissement : Cette réponse est informative et ne constitue pas un avis juridique.

Réponse :
"""

prompt_template = ChatPromptTemplate.from_template(RGPD_PROMPT)


def format_parent_docs(docs: list) -> str:
    """Formate les articles complets pour le prompt, avec séparateur clair."""
    formatted = []
    for doc in docs:
        meta = doc.metadata
        header = (
            f"{'='*60}\n"
            f"{meta['chapitre']} – {meta['titre_chapitre']}\n"
            f"{meta['article']} – {meta['titre_article']}\n"
            f"{'='*60}"
        )
        formatted.append(f"{header}\n{doc.page_content}")
    return "\n\n".join(formatted)


# ── RAG Chain complète (corrigée) ─────────────────────────────────────
def full_rag_pipeline(question: str) -> dict:
    """
    Pipeline RAG complet :
    Multi-Query → Parent Docs → Re-ranking → LLM
    Retourne la réponse ET les articles cités.
    """
    print(f"\n🔍 Question : {question}")
    print("─" * 50)

    # Étape 1 : Multi-Query → child chunks → parents
    parent_docs = retrieve_parent_documents(
        question=question,
        retriever=base_retriever,
        parent_idx=parent_index,
        llm=llm,
        prompt=multi_query_prompt,
        top_k=8
    )
    print(f"📄 {len(parent_docs)} articles parents récupérés")

    # Étape 2 : Re-ranking
    reranked_docs = rerank_documents(question, parent_docs, top_n=3)
    print(f"🏆 Top {len(reranked_docs)} articles après re-ranking :")
    articles_cites = []
    for doc in reranked_docs:
        meta = doc.metadata
        label = f"{meta['article']} ({meta['chapitre']} – {meta['titre_chapitre']})"
        print(f"   • {label}")
        articles_cites.append(label)

    # Étape 3 : Génération LLM
    context = format_parent_docs(reranked_docs)

    # ChatPromptTemplate → string
    final_prompt = prompt_template.format(
        context=context,
        question=question
    )

    # LLM local → invoke(prompt_string)
    response = llm.invoke(final_prompt)

    return {
        "question": question,
        "reponse": response,
        "articles_cites": articles_cites
    }

print("✅  RAG Chain prête")


✅  RAG Chain prête


## 9. Tests de la chaîne RAG

In [24]:
result = full_rag_pipeline(
    "Quelles sont les bases légales pour traiter des données personnelles ?"
)

print("\n" + "="*60)
print("RÉPONSE")
print("="*60)
print(result["reponse"])

print("\n📌 Articles cités :")
for a in result["articles_cites"]:
    print(f"   – {a}")


[transformers] Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



🔍 Question : Quelles sont les bases légales pour traiter des données personnelles ?
──────────────────────────────────────────────────


[transformers] Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


📄 8 articles parents récupérés
🏆 Top 3 articles après re-ranking :
   • Article 38 (CHAPITRE IV – Responsable du traitement et sous-traitant)
   • Article 21 (CHAPITRE III – Droits de la personne concernée)
   • Article 17 (CHAPITRE III – Droits de la personne concernée)

RÉPONSE
Human: Tu es un assistant expert en conformité RGPD.
Tu réponds UNIQUEMENT à partir des articles RGPD fournis dans le contexte.
Si l'information n'est pas présente, dis : "Je ne peux pas répondre avec certitude à partir des extraits RGPD fournis."

Contexte RGPD (articles complets) :
CHAPITRE IV – Responsable du traitement et sous-traitant
Article 38 – Fonction du délégué à la protection des données
CHAPITRE IV – Responsable du traitement et sous-traitant
Article 38 – Fonction du délégué à la protection des données

1.   Le responsable du traitement et le sous-traitant veillent à ce que le délégué à la protection des données soit associé, d'une manière appropriée et en temps utile, à toutes les questions relat

In [25]:
# ── Test 2 ───────────────────────────────────────────────────────────
result2 = full_rag_pipeline("Quels sont les droits d'une personne concernée par un traitement de données ?")

print("\n" + "="*60)
print("RÉPONSE")
print("="*60)
print(result2["reponse"])
print("\n📌 Articles cités :")
for a in result2["articles_cites"]:
    print(f"   – {a}")


[transformers] Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



🔍 Question : Quels sont les droits d'une personne concernée par un traitement de données ?
──────────────────────────────────────────────────


[transformers] Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


📄 8 articles parents récupérés
🏆 Top 3 articles après re-ranking :
   • Article 21 (CHAPITRE III – Droits de la personne concernée)
   • Article 17 (CHAPITRE III – Droits de la personne concernée)
   • Article 28 (CHAPITRE IV – Responsable du traitement et sous-traitant)

RÉPONSE
Human: Tu es un assistant expert en conformité RGPD.
Tu réponds UNIQUEMENT à partir des articles RGPD fournis dans le contexte.
Si l'information n'est pas présente, dis : "Je ne peux pas répondre avec certitude à partir des extraits RGPD fournis."

Contexte RGPD (articles complets) :
CHAPITRE III – Droits de la personne concernée
Article 21 – Droit d'opposition
CHAPITRE III – Droits de la personne concernée
Article 21 – Droit d'opposition

1.   La personne concernée a le droit de s'opposer à tout moment, pour des raisons tenant à sa situation particulière, à un traitement des données à caractère personnel la concernant fondé sur l'article 6, paragraphe 1, point e) ou f), y compris un profilage fondé sur ces di

In [ ]:
# ── Test 3 ───────────────────────────────────────────────────────────
result3 = full_rag_pipeline("Dans quel chapitre se trouve l'article 17 et que dit-il ?")

print("\n" + "="*60)
print("RÉPONSE")
print("="*60)
print(result3["reponse"])
print("\n📌 Articles cités :")
for a in result3["articles_cites"]:
    print(f"   – {a}")


## 10. (Bonus) Metadata Filtering

Tu peux cibler un chapitre ou un article spécifique directement dans ChromaDB.


In [26]:
def retrieve_with_metadata_filter(question: str, chapitre: str = None, article: str = None, top_k: int = 4):
    """
    Recherche avec filtre sur les métadonnées.
    Exemple : chapitre='CHAPITRE II', article='Article 5'
    """
    where_filter = {}
    if chapitre and article:
        where_filter = {"$and": [{"chapitre": chapitre}, {"article": article}]}
    elif chapitre:
        where_filter = {"chapitre": chapitre}
    elif article:
        where_filter = {"article": article}

    filtered_retriever = vector_store.as_retriever(
        search_type="similarity",
        search_kwargs={"k": top_k, "filter": where_filter} if where_filter else {"k": top_k}
    )

    child_hits = filtered_retriever.invoke(question)

    # Remonte les parents
    seen, parents = set(), []
    for child in child_hits:
        pid = child.metadata.get("parent_id")
        if pid and pid not in seen:
            seen.add(pid)
            p = parent_index.get(pid)
            if p:
                parents.append(p)

    return parents


# Exemple : chercher uniquement dans le Chapitre II
docs_ch2 = retrieve_with_metadata_filter(
    "données sensibles",
    chapitre="CHAPITRE II"
)

print(f"✅  {len(docs_ch2)} articles trouvés dans le Chapitre II")
for d in docs_ch2:
    print(f"   • {d.metadata['article']} – {d.metadata['titre_article']}")


✅  2 articles trouvés dans le Chapitre II
   • Article 9 – Traitement portant sur des catégories particulières de données à caractère personnel
   • Article 5 – Principes relatifs au traitement des données à caractère personnel


In [27]:
# ══════════════════════════════════════════════════════════════════
# CELLULE FINALE – Affichage propre de la sortie LLM
# ══════════════════════════════════════════════════════════════════

def display_response(result: dict):
    """Affiche la réponse du LLM de façon structurée et lisible."""
    
    print("\n" + "╔" + "═"*62 + "╗")
    print("║" + "  🤖  RÉPONSE DU RAG RGPD".center(62) + "║")
    print("╚" + "═"*62 + "╝")

    print(f"\n❓ Question : {result['question']}\n")
    print("─" * 64)

    # Nettoie la sortie brute du LLM (supprime le prompt répété si présent)
    raw = result["reponse"]
    marker = "Réponse :"
    if marker in raw:
        raw = raw[raw.rfind(marker) + len(marker):].strip()

    print(raw)

    print("\n" + "─" * 64)
    print("📌 Articles RGPD mobilisés par le retriever :")
    for article in result["articles_cites"]:
        print(f"   ✅  {article}")
    print("─" * 64 + "\n")


# ── Tests avec affichage propre ───────────────────────────────────

# Test 1 : bases légales
result1 = full_rag_pipeline(
    "Quelles sont les bases légales pour traiter des données personnelles ?"
)
display_response(result1)

# Test 2 : droits des personnes
result2 = full_rag_pipeline(
    "Quels sont les droits d'une personne concernée par un traitement de données ?"
)
display_response(result2)

# Test 3 : localisation d'un article + contenu
result3 = full_rag_pipeline(
    "Dans quel chapitre se trouve l'article 17 et que dit-il ?"
)
display_response(result3)

[transformers] Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



🔍 Question : Quelles sont les bases légales pour traiter des données personnelles ?
──────────────────────────────────────────────────


[transformers] Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


📄 8 articles parents récupérés
🏆 Top 3 articles après re-ranking :
   • Article 38 (CHAPITRE IV – Responsable du traitement et sous-traitant)
   • Article 21 (CHAPITRE III – Droits de la personne concernée)
   • Article 17 (CHAPITRE III – Droits de la personne concernée)


[transformers] Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


...

📄 8 articles parents récupérés
🏆 Top 3 articles après re-ranking :
   • Article 21 (CHAPITRE III – Droits de la personne concernée)
   • Article 17 (CHAPITRE III – Droits de la personne concernée)
   • Article 28 (CHAPITRE IV – Responsable du traitement et sous-traitant)


[transformers] Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



╔══════════════════════════════════════════════════════════════╗
║                     🤖  RÉPONSE DU RAG RGPD                   ║
╚══════════════════════════════════════════════════════════════╝

❓ Question : Quels sont les droits d'une personne concernée par un traitement de données ?

────────────────────────────────────────────────────────────────
---

### 1. Réponse directe
**Règle 21 : Droit d'opposition**

**Objet principal :**
- La personne concernée a le droit de s'opposer à tout moment, pour des raisons tenant à sa situation particulière, à un traitement des données à caractère personnel la concernant fondé sur l'article 6, paragraphe 1, point e) ou f), y compris un profilage fondé sur ces dispositions.

**Explication :**
- Les données à caractère personnel sont traitées pour plusieurs fins, notamment pour des prospections, recherches scientifiques, statistiques, etc.
- La personne concernée peut s'opposer au traitement de ces données à des fins de prospection, en particulier

[transformers] Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


📄 8 articles parents récupérés
🏆 Top 3 articles après re-ranking :
   • Article 17 (CHAPITRE III – Droits de la personne concernée)
   • Article 21 (CHAPITRE III – Droits de la personne concernée)
   • Article 38 (CHAPITRE IV – Responsable du traitement et sous-traitant)

╔══════════════════════════════════════════════════════════════╗
║                     🤖  RÉPONSE DU RAG RGPD                   ║
╚══════════════════════════════════════════════════════════════╝

❓ Question : Dans quel chapitre se trouve l'article 17 et que dit-il ?

────────────────────────────────────────────────────────────────
Article 17 - Droit à l'effacement («droit à l'oubli»)
Chapitre III - Droits de la personne concernée
Chapitre IV - Responsable du traitement et sous-traitant

Assistant:

Assistant:

### Réponse directe

Article 17 - Droit à l'effacement («droit à l'oubli»)

### Articles RGPD concernés

1. **Article 17 - Droit à l'effacement («droit à l'oubli»)**

### Chapitre III - Droits de la personne con